# 🧠 Colz AI — Training Notebook

**Kya hai yeh?**  
Yeh ek GPT-style AI hai jo pure Python mein scratch se banaya gaya hai.  
Koi PyTorch nahi, koi TensorFlow nahi — sirf Python + NumPy.

**Status:** Code ready hai ✅ — model abhi train ho raha hai 🔄  
Trained hone ke baad AI normal conversations kar sakta hai.

**GitHub:** https://github.com/niteenmaurya/Colz

In [ ]:
# ─── Step 1: GPU check ───────────────────────────────────────────────────────
# Colab ka free GPU mil raha hai ya nahi — yahan pata chalega
# Agar GPU mile toh training fast hogi

import torch
if torch.cuda.is_available():
    print(f"✅ GPU ready: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  GPU nahi mila — CPU pe chalega (thoda slow)")
    print("   Runtime > Change runtime type > T4 GPU select karo")

In [ ]:
# ─── Step 2: Repo clone ──────────────────────────────────────────────────────
# GitHub se latest code download karo
# Agar pehle se cloned hai toh pull karke update kar lo

import os

if os.path.exists('Colz'):
    print("📁 Repo pehle se hai — update kar raha hoon...")
    %cd Colz
    !git pull
else:
    print("📥 Repo clone ho raha hai...")
    !git clone https://github.com/niteenmaurya/Colz.git
    %cd Colz

print("✅ Code ready!")
!ls -la

In [ ]:
# ─── Step 3: Dependencies ────────────────────────────────────────────────────
# NumPy Colab mein already hai
# Flask server ke liye hai — training mein zaroorat nahi

!pip install numpy --quiet
print("✅ Dependencies ready!")

In [ ]:
# ─── Step 4: Data check ──────────────────────────────────────────────────────
# Training data kitna bada hai — pata karo
# Jitna zyada data utna better AI

with open('data.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"📊 Training data size : {len(text):,} characters")
print(f"📊 Unique characters  : {len(set(text))}")
print(f"\n📝 Sample (pehle 200 chars):")
print(text[:200])

In [ ]:
# ─── Step 5: Training config ─────────────────────────────────────────────────
# Model ka size aur training settings yahan set karo
# Zyada epochs = better AI lekin zyada time

EPOCHS   = 3000   # 3000 = achha output (~30 min on Colab GPU)
SEQ_LEN  = 64     # context window — kitne characters ek baar dekhega
LR       = 3e-4   # learning rate — kitna fast seekhega
LOG_EVERY = 100   # har 100 epochs mein loss print karo

print(f"⚙️  Epochs    : {EPOCHS}")
print(f"⚙️  Seq len   : {SEQ_LEN}")
print(f"⚙️  Learn rate: {LR}")
print("")
print("💡 Tip: Fast test ke liye EPOCHS = 500 rakho")
print("💡 Tip: Best quality ke liye EPOCHS = 5000 karo")

In [ ]:
# ─── Step 6: TRAIN ───────────────────────────────────────────────────────────
# Yahan actual training hoti hai
# Loss neeche girna chahiye — matlab AI seekh raha hai
# Loss 2.0 se neeche aaye toh achha output milega

from tokenizer import CharTokenizer
from transformer import GPT
from trainer import Trainer
from inference import generate
import pickle

# Tokenizer banao
tok = CharTokenizer()
tok.build_vocab(text)
tok.save('vocab.json')
print(f"📖 Vocab size: {tok.vocab_size} unique characters")

all_ids = tok.encode(text)

# Model banao
model = GPT(
    vocab_size  = tok.vocab_size,
    d_model     = 128,    # embedding size (bada = better lekin slow)
    n_heads     = 4,      # attention heads
    n_layers    = 3,      # transformer layers
    d_ff        = 512,    # feedforward size
    max_seq_len = SEQ_LEN,
)
print(f"🧠 Model parameters: {model.count_parameters():,}")
print(f"🚀 Training shuru ho rahi hai...")
print("-" * 50)

# Train karo
trainer = Trainer(model, lr=LR)
trainer.train(all_ids, seq_len=SEQ_LEN, epochs=EPOCHS, log_every=LOG_EVERY)

print("-" * 50)
print("✅ Training complete!")

In [ ]:
# ─── Step 7: Test karo ───────────────────────────────────────────────────────
# Training ke baad AI kya bol raha hai — dekho

test_prompts = [
    "hello",
    "how are you",
    "i am feeling",
    "what is your name",
    "i want to",
]

print("🗣️  AI ka output:\n")
for prompt in test_prompts:
    out = generate(model, tok, prompt, max_new=60, strategy='top_k', k=5)
    print(f"  Prompt  : {prompt}")
    print(f"  Output  : {out}")
    print()

In [ ]:
# ─── Step 8: Save + Download ─────────────────────────────────────────────────
# Trained model save karo aur download karo
# brain.pkl = AI ka trained dimag
# vocab.json = AI ki dictionary
# Dono files colz/ folder mein daalo apne PC pe

with open('brain.pkl', 'wb') as f:
    pickle.dump(model, f)

print("💾 brain.pkl save ho gaya!")
print("")
print("📥 Download ho raha hai...")

from google.colab import files
files.download('brain.pkl')
files.download('vocab.json')

print("")
print("✅ Done! Dono files download ho gayi.")
print("📌 Inhe C:\\Users\\nitee\\colz\\ mein daalo")
print("📌 Phir: cd web && python server.py")